# Strategy Router baseline — training, validation, your own queries

Track A of the router plan (SPEC d45/d46). Three stages, each one yours to read:

1. **Training** — fit the two logistic binaries on the decisive rows and look at which features drive each route.
2. **Validation** — score the router on held-out rows against the constant routes and the oracle ceiling.
3. **Your own queries** — route arbitrary text through the fitted model.

**Prerequisites**
- `data/route_labels/labels.parquet` and `data/feature_table/catalog.parquet` on disk.
- Stage 3 calls the taxonomy extractor, which needs spaCy: `poetry run python -m spacy download en_core_web_sm`.
- The optional full ablation downloads `multilingual-e5-small` (~470MB).

In [1]:
%load_ext autoreload
%autoreload 2

## 1 — Load the substrate

`exp.load()` joins the route labels to the 57 query features. `decisive_rows` keeps only the rows where one route clearly won (runner-up missed rank 1) — what the model trains and is scored on.

In [2]:
import pandas as pd
from IPython.display import display

from hybrid_search_rrf_dataset.router import (
    RouterExperiment,
    StrategyRouter,
    QueryEncoder,
    Representation,
    decisive_rows,
)

exp = RouterExperiment()
data = exp.load()
n_features = sum('.' in c for c in data.columns)
print(f'labelled rows: {len(data):,}  |  query features: {n_features}')

dec = decisive_rows(data)
print(f'decisive rows (trainable substrate): {len(dec):,}')
dec['winner'].value_counts().rename('decisive rows by winning route')

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


labelled rows: 24,338  |  query features: 57
decisive rows (trainable substrate): 2,510


winner
dense_only     1858
sparse_only     507
pure_rrf        145
Name: decisive rows by winning route, dtype: int64

## 2 — Training

Fit two one-vs-rest logistic models on the decisive rows of the training split: one for 'is this a dense win?', one for 'is this a sparse win?'. Thresholds are then tuned so ambiguous queries fall through to rrf. The coefficient tables show which features each binary leans on (weights are on standardised features, so magnitudes compare directly).

In [3]:
# Protocol (i): 80% of each lane trains, 20% is held out.
train, test = exp.split('random_within_lane')
print(f'train rows: {len(train):,}  |  held-out rows: {len(test):,}')

router = StrategyRouter(Representation.ENGINEERED).fit(train)
router.tune_thresholds(train)
print('tuned thresholds (dense, sparse):', tuple(round(t, 3) for t in router.thresholds))

train rows: 19,471  |  held-out rows: 4,867
tuned thresholds (dense, sparse): (0.45, 0.9)


In [4]:
coef = router.coefficients()
print('features pushing hardest toward DENSE:')
display(coef.sort_values('dense_weight', ascending=False).head(10).round(3))
print('features pushing hardest toward SPARSE:')
display(coef.sort_values('sparse_weight', ascending=False).head(10).round(3))

features pushing hardest toward DENSE:


,feature,dense_weight,sparse_weight
1,length.length_chars,1.037,-1.091
15,stopword_ratio.stopword_ratio,0.606,-0.494
48,structured_identifiers.social_handle,0.560,-0.571
56,syntactic_depth.statement_count,0.362,-0.415
52,structured_identifiers.uuid,0.348,-0.315
11,sentence_markers.greeting,0.338,-0.325
37,structured_identifiers.http_status_code,0.299,-0.320
13,sentence_markers.negation,0.293,-0.331
12,sentence_markers.interjection,0.215,-0.227
54,structured_identifiers.version_string,0.143,-0.120


features pushing hardest toward SPARSE:


,feature,dense_weight,sparse_weight
2,length.length_words,-1.230,1.553
7,morphology.word_variation_share,-0.513,0.533
9,sentence_markers.acronym,-0.193,0.348
31,structured_identifiers.error_code_like,-0.279,0.248
6,logical_structures.temporal,-0.165,0.157
26,structured_identifiers.currency_amount,-0.165,0.150
17,structured_identifiers.alt_geocoding,-0.119,0.137
0,coordination.widest_list_size,-0.124,0.104
44,structured_identifiers.postal_code,-0.062,0.097
4,logical_structures.math_expression,-0.072,0.084


## 3 — Validation

`exp.run` fits on the training portion and reports the six-column table over the held-out decisive rows, for both protocols:

- `random_within_lane` — held-out queries from lanes the model has seen.
- `holdout_lane` — the rarb-math lane held out whole (a collection never seen).

`headroom_captured` is where the router lands between the best constant route (0.0) and the oracle ceiling (1.0). The numbers are yours to read.

In [5]:
cols = [
    'protocol', 'representation', 'n_test_decisive',
    'const_dense_only', 'const_pure_rrf', 'const_sparse_only',
    'oracle', 'router', 'headroom_captured', 't_dense', 't_sparse',
]
result = exp.run(representations=[Representation.ENGINEERED])
result[cols].round(3)

holdout_lane·engineered: 100%|██████████| 2/2 [00:00<00:00, 22.17it/s, headroom=0.009, n=586]


,protocol,representation,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured,t_dense,t_sparse
0,random_within_lane,engineered,497,0.738,0.225,0.222,0.973,0.678,-0.251,0.45,0.85
1,holdout_lane,engineered,586,0.604,0.271,0.360,1.000,0.608,0.009,0.35,0.90


In [13]:
# Full three-representation ablation (engineered / e5 embedding / both).
# Uncomment to run — the first call downloads multilingual-e5-small (~470MB)
# and embeds every query once (cached to data/route_labels/e5_embeddings.parquet),
# so later runs are fast.

exp_full = RouterExperiment(encoder=QueryEncoder())
exp_full.run()[cols].round(3)

holdout_lane·both: 100%|██████████| 6/6 [02:42<00:00, 27.06s/it, headroom=0.008, n=586]      


,protocol,representation,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured,t_dense,t_sparse
0,random_within_lane,engineered,497,0.738,0.225,0.222,0.973,0.678,-0.251,0.45,0.85
1,random_within_lane,embedding,497,0.738,0.225,0.222,0.973,0.700,-0.158,0.45,0.90
2,random_within_lane,both,497,0.738,0.225,0.222,0.973,0.707,-0.129,0.45,0.90
3,holdout_lane,engineered,586,0.604,0.271,0.360,1.000,0.608,0.009,0.35,0.90
4,holdout_lane,embedding,586,0.604,0.271,0.360,1.000,0.602,-0.004,0.20,0.90
5,holdout_lane,both,586,0.604,0.271,0.360,1.000,0.607,0.008,0.15,0.90


## 4 — Test on your own queries

Fit on all decisive rows, then route whatever you type. This is the serving path: `predict` extracts the query's features inline and applies the same rule. Needs `en_core_web_sm` (spaCy).

In [4]:
served = StrategyRouter(Representation.ENGINEERED).fit(data, all_rows=True, max_class_share=0.45)
served.tune_thresholds(data)

my_queries = [
    'what are the side effects of DHA',
    'python sort list of dicts by key',
    'CVE-2021-44228 log4j remote code execution',
    'how do mRNA vaccines work',
    'SELECT * FROM users WHERE id = 42',
]
for q in my_queries:
    print(f'{served.predict(q)!s:12s}  {q}')

dense_only    what are the side effects of DHA
pure_rrf      python sort list of dicts by key
pure_rrf      CVE-2021-44228 log4j remote code execution
pure_rrf      how do mRNA vaccines work
dense_only    SELECT * FROM users WHERE id = 42


In [5]:
# Type your own query:
served.predict('#ABBSSS')

<StrategyName.DENSE_ONLY: 'dense_only'>

In [6]:
served.predict('Looking for qdrant_client.http.models.FormulaQuery')

<StrategyName.PURE_RRF: 'pure_rrf'>

In [7]:
served.explain('http://localhost.com')

{'query': 'http://localhost.com',
 'route': <StrategyName.PURE_RRF: 'pure_rrf'>,
 'p_dense': 0.14422330101849856,
 'p_sparse': 0.8262695529560404,
 't_dense': 0.45000000000000007,
 't_sparse': 0.9,
 'dense_fires': False,
 'sparse_fires': False}

In [11]:
served.predict('qdrant_client.http.models.FormulaQuery')

<StrategyName.DENSE_ONLY: 'dense_only'>

In [17]:
served.predict('CVE-1223')

<StrategyName.PURE_RRF: 'pure_rrf'>

In [26]:
served.predict('http://localhost.com')

<StrategyName.DENSE_ONLY: 'dense_only'>

In [18]:
from query_taxonomy.features import FeatureExtractor

fe = FeatureExtractor()

In [23]:
fe.resolve('http://localhost.com')

QueryFeatures(query_text='http://localhost.com', spans={<FeatureGroup.STRUCTURED_IDENTIFIERS: 'structured_identifiers'>: {'uri': [FeatureSpan(text='http://localhost.com', start=0, end=20)]}}, stats={<FeatureGroup.STATISTICAL_METRICS: 'statistical_metrics'>: {'length': [FeatureStat(name='length_words', value=3.0), FeatureStat(name='length_chars', value=20.0)], 'stopword_ratio': [FeatureStat(name='stopword_ratio', value=0.0)]}}, tfs={<FeatureGroup.STRUCTURED_IDENTIFIERS: 'structured_identifiers'>: {'uri': 1}})

In [22]:
from query_taxonomy.banks.tech import URIBank


uri_detector = URIBank()

uri_detector.compute("http://localhost.com")

[FeatureSpan(text='http://localhost.com', start=0, end=20)]

In [25]:
from hybrid_search_rrf_dataset.router import _extract_features

features = _extract_features(None, "http://localhost.com")
features


{'structured_identifiers.uri': 1,
 'length.length_words': 3.0,
 'length.length_chars': 20.0,
 'stopword_ratio.stopword_ratio': 0.0,
 'natural_language_signal.natural_language_share': 0.0,
 'morphology.word_variation_share': 0.0,
 'syntactic_depth.nesting_depth': 0.0,
 'syntactic_depth.statement_count': 1.0,
 'coordination.widest_list_size': 0.0}